[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/14_business_ai/45_ai_assisted_software_development.ipynb)

# 📓 Notebook 45 — AI-Assisted Software Development

> **Module:** Business AI in Practice · **Estimated time:** ~2 h · **Difficulty:** Intermediate

> 📍 **Do this early.** Although it lives in Module 14, the Git + GitHub + Copilot workflow here pays off across *every* notebook — the onboarding recommends it right after the Python basics (Module 1).

In 2024–2026, *how* code gets written changed more than in the previous twenty years. A working professional in 2026 expects to spend a significant fraction of their day directing an AI assistant rather than typing. This notebook is about that workflow — the tools, the habits, and most importantly the **critical-review discipline** that separates *AI-assisted productivity* from *AI-assisted bug-introduction*.

> 🏢 **Picking up Meridian's story.** In NB 44, Meridian — our ~400-person B2B SaaS company — settled on a **3-tier app** for its ticket-triage feature: an agent UI, a FastAPI middle tier with the classify logic, a Postgres store, and the LLM as an external call. Now its *one* engineer has to actually build it, fast, with an AI assistant doing much of the typing. This notebook is that engineer's playbook. Watch the 🏢 marker for how each habit shows up in their build.

> 🧭 **The mental model to carry through this notebook.** Think of the AI assistant as **a fast, confident junior who never says "I don't know."** It's brilliant at boilerplate and tireless at tests — but it will hand you plausible-looking nonsense with total confidence, so *you* are the senior reviewer, always. Two corollaries follow and recur all through the notebook: **(1) only delegate what you could review**, and **(2) review every diff before you trust it.** We'll come back to this junior in the recap.

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Pick** an IDE and AI-assistant combination that matches your work style.
2. **Use Git fluently** for the operations 90 % of working programmers do (commit / branch / pull request / merge conflict).
3. **Write prompts for code** that consistently produce reviewable, runnable output — not vague drafts.
4. **Critically review** AI-generated code, naming the four most common failure modes.
5. **Decide** when *not* to use the AI assistant.

**Prerequisites:** Modules 1–4 (you should already be writing Python comfortably). No prior Git knowledge assumed — §2 teaches what you need.

**Time budget:** ~80 minutes including the practical exercises.

## 1. The modern IDE landscape (2026)

Four mainstream options for Python work, each with different trade-offs:

| IDE | AI assistance | Best for | Note |
|---|---|---|---|
| **VS Code + GitHub Copilot** | Inline completions + Copilot Chat | the broadest user base; great Jupyter support; works on every OS | Free for students/teachers; otherwise ~$10/mo |
| **Cursor** | Chat is the *main* interface; multi-file refactors | rewriting larger chunks of existing code; *agentic* edits | VS Code fork, so muscle memory transfers |
| **PyCharm + JetBrains AI** | Inline completions + AI Assistant panel | large Python projects, refactor-heavy work | The most batteries-included for Python specifically |
| **Jupyter / JupyterLab** | Plugin: Jupyter AI; or use Cursor with Jupyter mode | exploration, teaching, this course's notebooks | Less suited for multi-file projects |

> 💡 **Practical recommendation for this course's learners.** If you're a beginner, *VS Code + Copilot* or *Cursor* are the most forgiving. If you're already at home in JetBrains, stay there — the AI feature parity is acceptable in 2026 and switching IDEs costs more than the AI buys.


### The two interaction modes — *completion* vs *chat*

Modern AI assistants present roughly two interfaces, and each suits a different task:

- **Inline completion** (the *grey-ghost-text* that appears as you type) — fast, low-effort, good for small repetitive code (loops, type annotations, boilerplate). Accept with `Tab`.
- **Chat** (a side panel where you type a request and the AI proposes a multi-line / multi-file change) — slower, higher-effort, good for *real* work: refactoring, implementing a new function, writing tests.

Most working programmers use both. A common rhythm:

```
  open file  →  chat: "add a function that does X"
             →  review the proposal critically
             →  accept / reject / tweak
             →  fill in surrounding code with inline completion
             →  chat: "add tests for X"
             →  review
             →  commit
```

The trap to avoid: using *only* inline completion. It's pleasant and feels fast, but it can't catch the bigger-picture mistakes (architectural drift, duplicated logic, missing edge cases). Chat is where real review happens.


---

### ✋ Quick exercise (~2 min) — Completion or chat?

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

You have two jobs on Meridian's triage feature today. **Task A:** add type hints to 15 small one-line helper functions. **Task B:** implement a brand-new `classify` function whose logic spans the FastAPI route and a helper module. For each task, pick **inline completion** or **chat**, and say why in one phrase.

**✍️ Your turn:** For Task A and Task B above, pick *inline completion* or *chat* and justify each in one phrase.

_Your answer:_

<details>
<summary>✅ <b>Solution</b></summary>

- **Task A → inline completion.** Type hints on small repetitive helpers are exactly the low-effort, boilerplate work the grey-ghost-text is fast at — accept with `Tab` and move on.
- **Task B → chat.** New logic that spans more than one place needs the higher-effort interface where real review happens; inline completion can't catch the bigger-picture mistakes (architectural drift, missing edge cases).

The rule of thumb from §1: completion for repetitive boilerplate, chat for any change that introduces new logic or touches more than one file.
</details>

## 2. Git basics — what 90% of professional work looks like

Git is not part of *this* course's installation requirements, but you will not get a job in 2026 without it. The good news: you only need a small subset of its commands to be productive.

The mental model: Git is a **time machine for your code that you control with snapshots called *commits***. Snapshots live on *branches*. You collaborate by sending other people your branch (a *pull request* / *merge request*) so they can review and merge it into the main branch.

```
      main branch ───●────●────●─────────●────●────●─────►
                              \                  /
                               ●───●───●───●──● ◄── your feature branch,
                                                    merged via a PR
```

### The 9 commands that cover 90% of daily work

```bash
git status                       # what's changed since last commit?
git diff                         # show the actual line-level diff
git add path/to/file.py         # stage a file for the next commit
git commit -m "short message"   # record a snapshot
git pull                         # fetch + integrate latest from remote
git push                         # send your commits to remote
git checkout -b new-feature      # create + switch to a new branch
git checkout main                # switch back to main
git merge new-feature            # merge the feature branch into the current branch
```

Plus one operation that scares everyone the first time:

```bash
# A merge conflict — Git can't decide which side wins for some lines.
# Git marks the conflict in the file like this:
#
#   <<<<<<< HEAD
#   your version of the line
#   =======
#   the other version of the line
#   >>>>>>> the-other-branch
#
# Open the file, delete the markers, keep what you want, save, then:
git add path/to/file.py
git commit            # finishes the merge
```

> ⚠️ **Common beginner mistake.** *Force-pushing* (`git push --force`) over the main branch destroys history other people have built on. The rule: never force-push to a shared branch; force-push only your own feature branch, and only when you're sure nobody else is working on it.


### 🔬 What actually happens — a *commit* is a snapshot, a *diff* is computed

§2 calls Git a "time machine ... with snapshots called commits." That metaphor is *literally true*, and you can build the mechanic in ~15 lines of standard-library Python. Two ideas do all the work:

1. **A commit stores a *snapshot*, addressed by the hash of its content.** Git doesn't store "you changed line 7." It stores the *whole file's bytes*, names that blob by its SHA hash, and a commit just points at that snapshot. Identical content → identical hash → stored once. This is why `git status` is instant: comparing two 40-char hashes tells you if anything changed without re-reading the files.
2. **A *diff* is not stored — it's computed on demand** by comparing two snapshots line-by-line. `git diff` runs a line-comparison algorithm (Git uses a Myers diff; Python's `difflib` uses a different one — Ratcliff/Obershelp — but both emit the same unified-diff format) over the old and new content *when you ask*. The `<<<<<<<` / `=======` / `>>>>>>>` merge-conflict markers you saw above are just that diff machinery admitting it can't pick a side.

```text
       commit A                 commit B
          │                        │
          ▼                        ▼
   ┌──────────────┐         ┌──────────────┐
   │ snapshot     │         │ snapshot     │
   │ hash=a1b2c3… │         │ hash=f4e5d6… │   ← whole content, named by SHA
   │ "def f(): 1" │         │ "def f(): 2" │
   └──────┬───────┘         └──────┬───────┘
          └────────►  diff  ◄───────┘
                   (computed NOW, not stored)
                   - def f(): 1
                   + def f(): 2
```

Below we build both halves offline with `hashlib` and `difflib` — no Git, no network — and prove the two properties that make the metaphor real.

**Half 1 — a content-addressed snapshot store.** A commit names its snapshot by a hash of the bytes. We mimic that with a `dict` keyed by `hashlib.sha1`. The payoff: storing the *same content twice* costs nothing (same key), and a one-character change produces a completely different address — exactly how Git deduplicates and detects change.

In [ ]:
import hashlib

class SnapshotStore:
    """A 10-line stand-in for how Git names commits: address content by its hash."""
    def __init__(self):
        self._objects = {}                     # hash -> content (Git calls these "blobs")

    def commit(self, content):
        h = hashlib.sha1(content.encode()).hexdigest()
        self._objects[h] = content             # idempotent: same content -> same key
        return h                               # the "commit id" you'd see in `git log`

store = SnapshotStore()

v1 = "def greet():\n    return 'hi'\n"
v2 = "def greet():\n    return 'hello'\n"      # one line changed

id1 = store.commit(v1)
id2 = store.commit(v2)
id1_again = store.commit(v1)                    # re-commit the EXACT same content

print("commit 1:", id1[:12])
print("commit 2:", id2[:12])
print("same content -> same id? ", id1 == id1_again)   # True: snapshots are deduplicated
print("any change -> new id?    ", id1 != id2)         # True: one char flips the whole hash
print("objects actually stored: ", len(store._objects))# 2, not 3 -> v1 stored once



**Half 2 — the diff is computed, not stored.** Nothing above recorded "line 2 changed." That information is *derived* by comparing two snapshots whenever you run `git diff`. `difflib.unified_diff` produces the very `- old` / `+ new` format you read in pull requests, and `difflib`'s conflict-finding is the same idea behind those `<<<<<<<` merge markers.

In [ ]:
import difflib

# Reconstruct the two snapshots from the store by their commit ids -- then diff THEM.
old = store._objects[id1].splitlines()   # lines without terminators (pairs with lineterm="")
new = store._objects[id2].splitlines()

diff = difflib.unified_diff(old, new, fromfile="greet@" + id1[:7],
                            tofile="greet@" + id2[:7], lineterm="")
print("=== what `git diff` computes on demand ===")
for line in diff:
    print(line)

# The SAME machinery underlies a merge conflict: when two branches changed the
# same line, Git can't auto-pick, so it emits both sides between markers.
print("\n=== how a conflict marker is produced ===")
ours   = "    return 'hello'"
theirs = "    return 'hey'"
print("<<<<<<< HEAD")
print(ours)
print("=======")
print(theirs)
print(">>>>>>> other-branch")



> 🧠 **Mental model.** A commit is a *named snapshot of whole content* (named by a hash of its bytes), and a diff is *computed by comparing two snapshots on demand* — neither is "a list of edits Git remembers." That single fact demystifies the everyday commands: `git status` is fast because it compares hashes, not files; identical content is never stored twice; `git diff` and the `<<<<<<<`/`=======`/`>>>>>>>` conflict markers are the *same* line-comparison machinery you just ran with `difflib`. When you review an AI-generated PR (§4), the "diff" you scrutinise is exactly this computed comparison between the last snapshot and the proposed one.

### Pull requests — the real unit of professional work

Once you have a feature branch with a few commits on it, you push it to GitHub / GitLab / Bitbucket and open a **pull request** (PR) — a request to merge your branch into `main`. A good PR has:

- A **short title** that reads like a sentence: *"Add retry-with-backoff to the LLM client."*
- A **2-paragraph description**: what changed, why, and what testing was done.
- **Small scope.** A 30-line PR gets reviewed in 5 minutes; a 3,000-line PR sits in someone's queue for a week.
- **Tests** (if the codebase has tests) or at least a screenshot of having run the new code.

AI-generated PRs are no exception — arguably *more* important to keep small, because the reviewer's main job is checking the AI didn't make subtle mistakes. A 2,000-line AI-generated PR is genuinely unreviewable; a 50-line one is fine.


---

### ✋ Quick exercise (~2 min) — From edit to pull request

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

You just edited `classify.py` while sitting on `main`. Using only the nine core commands from §2, write the sequence of `git` commands that moves this change onto a new feature branch and gets it ready to open a PR (branch → stage → snapshot → send to remote).

**✍️ Your turn:** Write the ordered list of `git` commands.

_Your answer:_

<details>
<summary>✅ <b>Solution</b></summary>

```bash
git checkout -b add-classify             # create + switch to a feature branch
git add classify.py                      # stage the change
git commit -m "Add classify endpoint"    # record a snapshot
git push                                  # send the branch to remote, then open the PR
```

Running `git status` / `git diff` before staging is a good habit to confirm *what* you're committing. Branching first keeps `main` clean and lets the change ship as a small, reviewable PR — §2's "real unit of professional work."
</details>

## 3. Prompt engineering for code — what works in 2026

Your confident junior is only as good as the brief you give it. A vague brief gets you a vague — and often wrong — answer, fast. So the highest-leverage skill in AI-assisted coding isn't typing; it's *briefing*. Five patterns that reliably produce reviewable, runnable output from a code-assistant LLM. None of them is fancy; the working programmer's habits are quietly *very* unfancy.

> 🏢 **Meridian's engineer at the keyboard.** Building the triage middle tier, Meridian's engineer doesn't type *"build me a ticket classifier."* They brief the junior the way they'd brief an actual junior — one function at a time, with the real context pasted in. The five patterns below are exactly the moves they make; watch them assemble the `classify` endpoint piece by piece.

### Pattern 1 — give the AI *what it can't see*

Chat assistants see only what's open / pinned in your IDE. If a function lives in another file, *paste the function* into the prompt. If a schema lives in a YAML file, paste the YAML. **The single biggest cause of bad AI code is hallucinated context** — the model invents a function signature because it couldn't see the real one.

**Bad prompt:** *"Update the user-fetch function to add caching."*

**Good prompt:** *"Here's the current `fetch_user` function in `users/api.py`: [paste]. Add caching using `functools.lru_cache`, max size 1000. Keep the existing signature."*

### Pattern 2 — ask for *one thing*, explicitly

A prompt with three asks tends to get three half-done answers. Better: one ask per turn, sequentially.

**Bad:** *"Write a function that fetches users, caches them, retries on failure, and logs everything."*

**Better (sequence):**
1. *"Write a function that fetches users from `https://api.example.com/users/{id}` using `requests`, returning a dict. Handle the HTTP-error case by raising."*
2. *"Add caching to the function above using `functools.lru_cache`. Keep the rest the same."*
3. *"Add retry-on-failure with exponential backoff. Use the `tenacity` library; max 3 attempts."*
4. *"Add `logging.info` calls at each step."*

The sequence costs you a few seconds; each turn produces something you can review and accept before moving on. The single-prompt version produces an opaque 60-line blob.


### Pattern 3 — be specific about *constraints*

Constraints that tend to matter and that the AI will otherwise pick at random:

- **Python version** ("using Python 3.11+ syntax").
- **External libraries to use / avoid** ("use `httpx`, not `requests`" or "standard library only").
- **Style** ("add type hints", "follow PEP 8", "keep functions under 30 lines").
- **Performance constraint** ("this runs on a 10M-row DataFrame — vectorise, don't loop").
- **Error semantics** ("raise on missing key, don't silently return `None`").

Each constraint takes a sentence; each one would otherwise cost an iteration cycle.


### Pattern 4 — ask for *tests*

After implementing a function, ask: *"Write `pytest` tests for the above function. Cover the happy path, the empty-input case, and one error case."*

Why: writing tests is the highest-value thing an AI assistant does. Tests are usually boring to write, easy to verify (you can run them), and they catch the AI's own subtle mistakes. They are also documentation of intent.

**The discipline:** every time you accept an AI-generated function, accept tests at the same time. Don't accept the function without them.


### Pattern 5 — ask for *alternatives* when designing

For design questions (not implementation), the most useful pattern is *"give me three approaches with trade-offs"*:

> *"I need to store ~10M chat-message records with full-text search. Give me three storage options with their trade-offs for our use case. We are a 5-person startup on AWS, currently using Postgres."*

The LLM will produce a comparable table (Postgres-FTS / Elastic / OpenSearch / Typesense / …). You pick. The value is not the AI's *opinion* — it's having an enumerated, comparable shortlist in 30 seconds instead of an afternoon of Googling.


---

### ✋ Quick exercise (~2 min) — Fix the brief

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

A teammate sends the assistant: *"Make the user-fetch function faster."* Rewrite this into **one** good prompt by applying **Pattern 1** (give it what it can't see) and **Pattern 3** (be specific about constraints). The function is `fetch_user(user_id)` and should stay standard-library-friendly.

**✍️ Your turn:** Rewrite the vague prompt into one specific, reviewable brief.

_Your answer:_

<details>
<summary>✅ <b>Solution</b></summary>

> *"Here's the current `fetch_user(user_id)` function: [paste]. It's slow when called in a loop. Add caching with `functools.lru_cache` (max size 1000), keep the existing signature, and add type hints. Standard library only."*

This pastes the real function so the model can't hallucinate its signature (Pattern 1), and pins concrete constraints — caching mechanism, signature, style, no new dependencies (Pattern 3). The result is reviewable in one pass instead of a guess you have to re-brief.
</details>

## 4. Critical review of AI-generated artefacts — the four failure modes

Here is where the *confident junior* metaphor earns its keep. The junior never flags its own mistakes — so if you don't know the four shapes those mistakes take, they ship. Knowing *how AI code fails* is the most underrated professional skill of 2026. Four distinct failure modes, each with its own tell.

### Failure mode A — *Hallucinated APIs*

The model invents a function, class, or argument that doesn't exist. *Looks* like real code, runs nowhere.

```python
# AI-generated:
from openai import OpenAI
client = OpenAI()
resp = client.chat.completions.create_with_retry(   # ← invented method
    model="gpt-5.5",
    messages=[...],
    max_retries=3,                                    # ← invented argument
)
```

**Detection:** run the code. If you can't run it, check the library's actual docs (one click in the IDE). Don't accept code you haven't either run *or* eyeballed against the library reference.

**Frequency:** much rarer in 2026 than in 2023, but still happens for less-popular libraries and for recent API changes.

### Failure mode B — *Plausible but wrong*

The code runs, but does the wrong thing. The most dangerous failure mode because there is no error message to catch it.

```python
# AI-generated: "weighted average of values"
def weighted_average(values, weights):
    return sum(values) / sum(weights)   # ← not the weighted average; just the ratio of sums
```

**Detection:** the only defence is tests. A 2-line test against a hand-computed example catches this in 10 seconds:

```python
assert weighted_average([10, 20], [1, 3]) == 17.5   # would fail with the above
```

**Mitigation:** Pattern 4 from §3 — *always* ask for tests with concrete numeric examples.


### Failure mode C — *Architectural drift*

The AI's code works, but doesn't fit the conventions of the surrounding codebase. Two real-life examples:

- The codebase uses `logging.info(...)` throughout; the AI's new code uses `print(...)`.
- The codebase has an established `Result` / `Either` pattern for error returns; the AI raises an exception that the surrounding code doesn't expect.

**Detection:** the diff looks too *novel* — the AI introduced a vocabulary that doesn't appear elsewhere in the codebase. A linter or a `grep` for the rest of the file catches most cases.

**Mitigation:** include *one similar function from the codebase* in your prompt so the model has the convention to copy.


### Failure mode D — *Silent over-confidence*

The AI hides edge cases under one-line defaults. The code looks elegant; the edge cases are bugs.

```python
# AI-generated: "safely parse the JSON response"
def parse(text):
    try:
        return json.loads(text)
    except Exception:
        return {}      # ← silently returns empty dict on ANY error, including a typo
                       #   in the upstream system that should have been a loud failure
```

**Detection:** look for `except Exception: pass` or `except Exception: return ...` patterns. They almost always hide bugs in the long run. So do *unjustified* `or default` patterns, broad catches without logging, and silent type coercions.

**Mitigation:** add a constraint to your prompt — *"raise on errors, don't silently catch."* Or fix it on review.


### The 60-second review checklist

Run through this for every AI-generated change before clicking *Accept* or merging the PR:

1. **Does it run?** Even just the import line.
2. **Are there tests?** If not, ask for them.
3. **Does it use the libraries / patterns we use elsewhere?** Grep the codebase for one similar function.
4. **Does it silently swallow errors?** Search the diff for `except Exception`, `or {}`, `or []`, `pass`.
5. **Does the code do what the prompt asked, no more no less?** A `weighted_average` shouldn't *also* log to disk.

60 seconds, catches 80% of the failures. The remaining 20% is what real testing and code review are for.

> 🏢 **Meridian catches one.** Running this checklist over the AI-drafted `classify` endpoint, Meridian's engineer hits item 4: the junior had wrapped the LLM call in `except Exception: return {"category": "other"}` — silently mislabelling every ticket as *other* the moment the provider hiccups (failure mode D). One line removed, one explicit `raise` added, one test that asserts an outage *fails loudly*. This is the senior-reviewer reflex the whole notebook is training — and it's exactly the kind of silent degradation NB 46 will harp on at production scale.

---

### ✋ Quick exercise (~2 min) — Name that failure mode

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The assistant hands you this helper for Meridian's triage service. Which of the **four failure modes** from §4 is it, and what is the one-line fix?

```python
def load_config(path):
    try:
        return json.load(open(path))
    except Exception:
        return {}
```

**✍️ Your turn:** Name the failure mode and give the one-line fix.

_Your answer:_

<details>
<summary>✅ <b>Solution</b></summary>

**Failure mode D — silent over-confidence.** The blanket `except Exception: return {}` swallows *every* error (a missing file, malformed JSON, a permissions problem) and hands back an empty config as if nothing were wrong — a bug that surfaces far from its cause.

**Fix:** don't catch it; let a bad config fail loudly:
```python
def load_config(path):
    with open(path) as f:
        return json.load(f)
```

This is item 4 of the 60-second checklist — *"does it silently swallow errors?"* — and the same reflex that caught Meridian's `except Exception: return {"category": "other"}`.
</details>

## 5. When *not* to use the AI assistant

The confident junior is fast and cheap — which makes it tempting to hand it everything. But there are tasks where handing it the work costs you more than doing it yourself. Three situations where the cost of using the assistant exceeds the benefit:

**(a) Learning a new concept for the first time.** If your goal is to understand `asyncio` so you can use it for years, having the AI write the code defeats the point. The 30 minutes of confusion is the learning. Use the AI to *explain*, not to *generate* — *"Explain how `asyncio.gather` differs from a thread pool, with a small example"* is fine; *"Write me an async data pipeline"* skips the learning.

**(b) Security-sensitive code.** Auth, crypto, input sanitisation, password hashing. The AI's defaults are often *just* wrong enough to introduce a subtle vulnerability. Use a vetted library (`bcrypt`, `argon2`, your platform's `oauth2` lib) rather than letting the AI roll its own.

**(c) Anything you don't understand well enough to review.** This is the asymmetric one: if you can't tell whether the output is right, the AI saving you 10 minutes today costs you a debugging session next week. The honest move is to *not* use the AI for that task — and spend the 10 minutes reading instead.

> 🧭 **Mental model, sharpened.** This is corollary (1) of the confident junior, stated plainly: **only delegate what you could review.** AI assistants are most valuable when you could write the code yourself but would rather not. They are least valuable — actively dangerous, even — when you couldn't review the code if it were handed to you. A junior you can't check is a liability, not leverage.

## 6. How this course is set up to teach this workflow

Several earlier notebooks practise the pieces this notebook puts together:

- **NB 22 — AI workflows.** Shows the classification / extraction patterns the AI is well-suited to.
- **NB 26 — AI evaluation & observability.** How to monitor AI output for the kind of drift this notebook calls out.
- **NB 39 — From notebook to project.** The packaging and testing discipline that makes any of the above shippable.
- **NB 27–30 — Building AI POCs (Module 7).** Where you *apply* this workflow hands-on: set up Copilot Agent Mode and build real prototypes. This notebook is the *judgement*; Module 7 is the *practice*.

**Recommended sequence if you want to practise this notebook's content end-to-end:** install Cursor (or VS Code + Copilot), clone this course's repo, open NB 39's `costkit_demo` example, and try to add a new module + tests + a small CLI for it using *only* AI assistance. Then PR it to yourself (your own repo, branch → PR → merge). That's ~2 hours of work and you will hit every failure mode in §4 at least once.


## 🧪 Practice exercises

Mostly hands-on — these expect you to use a real AI assistant in a real IDE.

### Exercise 1 — ⭐ Run the 60-second checklist

Take this AI-generated function. Run the 60-second checklist from §4 against it. Identify at least two issues.

```python
def get_user_age(users, email):
    """Return the age of the user with the given email, or None."""
    for u in users:
        if u.get('email') == email:
            return datetime.now().year - u['birth_year']
    return None
```

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Issues a 60-second review catches:**

1. **Missing import.** `datetime` is used but not imported. Would fail at `NameError`. (Failure mode A — assumed/hallucinated context.)
2. **`KeyError` on `u['birth_year']`** if the user dict is missing that field. The code defends against missing `email` (uses `.get`) but not missing `birth_year` (uses `[]`). (Failure mode D — inconsistent error handling.)
3. **No tests.** Should be requested before accepting.
4. **Possibly wrong domain logic:** `year - birth_year` is age in *some* year — wrong by up to 1 depending on whether the user's birthday has passed. The correct version uses a date comparison. (Failure mode B — plausible but wrong.)

*Two out of four is enough to pass; identifying all four is excellent.*
</details>

### Exercise 2 — ⭐⭐ Sequence a prompt

You need a function that: (a) reads tickets from a CSV, (b) classifies each by topic using the course's `MockLLM`, (c) writes the results to a new CSV with the original columns + a `topic` column, (d) logs progress every 100 rows. Write the **four sequential prompts** you would use rather than a single big one. Show how each builds on the last.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Suggested prompt sequence:**

1. *"Write a Python function `load_tickets(csv_path)` that uses pandas to read a CSV and returns a DataFrame. Require a `text` and `id` column; raise `ValueError` if either is missing."*

2. *"Here's the function above: [paste]. Now write `classify_topic(text, llm)` that takes one ticket text and one LLM client; returns a topic string. Use a system prompt asking the model to reply with just the topic word from {billing, tech, feature, other}."*

3. *"Compose the two: write `classify_all(tickets, llm)` that takes the DataFrame from (1) and the function from (2), and returns a new DataFrame with an extra `topic` column. Use `df.apply` along axis=1 — no for-loop."*

4. *"Add `logging.info` calls in `classify_all` that report every 100 rows: how many done, how many remaining, estimated time-to-completion based on the per-row latency seen so far. Use the existing logger; don't print."*

Each prompt is short, self-contained, and produces something you can run + review before moving on. The single-prompt version would have produced a 40-line blob that probably has one of failure modes B / C / D somewhere in it.
</details>

### Exercise 3 — ⭐⭐ Write a 30-line PR description

You added the `classify_all` function from Exercise 2 to a real project. Draft the PR description (3 paragraphs max) that a human reviewer would actually find useful. Include the *what*, *why*, *how-tested*, and *risks*.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Example PR description:**

> **`classify_all`: batch classify tickets via the existing MockLLM**
>
> *What.* Adds `classify_all(tickets_df, llm) -> DataFrame` in `support/classify.py`. The function takes a tickets DataFrame (must have `id` and `text` columns) and returns the same DataFrame with an extra `topic` column. Topic is one of `{billing, tech, feature, other}`. Helper `classify_topic(text, llm)` is unit-tested in isolation.
>
> *Why.* The triage script (`scripts/nightly_triage.py`) currently classifies tickets one-by-one with hand-rolled logic. This function will replace lines 42–67 of that script next sprint; this PR adds the function and tests, no behaviour change yet.
>
> *How-tested.* Two new pytest cases in `tests/support/test_classify.py`: one happy-path on 5 sample tickets (asserts each gets one of the four valid topics); one error-path that passes a DataFrame missing the `text` column (asserts `ValueError`). Manually ran on the existing 3-day backlog (~600 tickets) with `MockLLM(seed=0)` — completed in 4 s, all rows classified.
>
> *Risk.* MockLLM behaviour is deterministic; classification quality is bounded by the keyword-based rules. The follow-up PR (#241) swaps the `MockLLM` for the production `OpenAILLM` and adds a per-row token-cost log.

**What makes this good:** all four sections present; mentions the *follow-up* PR explicitly so the reviewer knows what's *not* in scope; concrete test numbers (5, 600); risk acknowledges what the PR doesn't cover.
</details>

### Exercise 4 — ⭐⭐ Decide *not* to use the AI

Pick three programming tasks from your own work (real or hypothetical). For each, decide whether to use AI assistance, and **defend the decision** in one sentence. At least one should be a *no*.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Example for a backend engineer:**

1. *Writing a regex to validate a German tax-ID format.* **Yes, use AI.** The regex is well-known, the validation is straightforward, the AI's output can be tested against 5 known-good and 5 known-bad examples in 30 seconds. Saves 20 minutes of regex-tweaking.

2. *Rotating the OAuth signing key for the customer SSO integration.* **No, do not use AI.** Cryptographic operations have non-obvious correctness requirements; the AI is statistically likely to produce *plausible-but-wrong* code (failure mode B); the fallout from getting it wrong is everyone's session being invalidated or — worse — sessions being signable with the old key. Use the platform's documented key-rotation procedure manually.

3. *Learning to use `asyncio.TaskGroup` for the first time, for a small parallel-fetch helper.* **No, do not generate; yes to explain.** The 45 minutes of "why doesn't it work the way I expect" *is* the learning. I'll use AI to explain `TaskGroup` semantics and to review my code afterwards — but I'll write the code myself.

*The point of the exercise is the deliberation, not the specific examples.*
</details>

### Exercise 5 — ⭐⭐ Debug me 🐞 (review-style)

Read this AI-generated function. Find at least three issues and propose a fix for each.

```python
def fetch_and_save(url, output_path):
    """Download a JSON document and save it locally."""
    import requests
    response = requests.get(url)
    data = response.json()
    with open(output_path, 'w') as f:
        f.write(str(data))
    return True
```

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Issues:**

1. **Import inside the function.** Imports belong at the top of the module so they're visible and so the cost is paid once. *Fix:* move `import requests` to the top.

2. **No timeout on `requests.get`.** A hung server hangs the function forever. *Fix:* `requests.get(url, timeout=10)`.

3. **No HTTP error handling.** A 500 response will silently make `response.json()` raise (or worse — return `{'error': ...}` that gets written to disk as if it were the data). *Fix:* `response.raise_for_status()` before `.json()`.

4. **`f.write(str(data))` writes Python's `repr` to disk, not JSON.** A dict written via `str()` produces something like `{'key': 'value'}` with single quotes — not valid JSON, can't be re-parsed. *Fix:* `json.dump(data, f)`.

5. **Returning `True` is meaningless.** Either return the data, return `None`, or remove the return. (Failure mode D — confident-looking but useless.)

6. **No tests.** Ask the AI for `pytest` cases that mock `requests.get` and verify the file is written as valid JSON.

*Catching 3 of the 6 is a passing review; catching 5+ is excellent.*
</details>

## 🧠 Stretch exercises

### Stretch exercise A — ⭐⭐⭐ Build a small project with AI assistance, then review your own PR

Open a fresh repo locally. Using *only AI assistance* (no manually-written code), build a CLI tool that converts an input CSV into a Markdown table. ~50 LOC total. Then:

1. Open the diff. Run the 60-second checklist from §4 against your own code.
2. Write a real PR description (Exercise 3 format).
3. Identify which of the four failure modes occurred at least once during your session.

The exercise is the *self-review*. Most professional time spent with AI assistants is reviewing your own AI-generated code; doing it deliberately once builds the habit.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

There's no single answer — the exercise is to *do it*. A few signals you did it well:

- You hit at least one of the four failure modes (most learners hit B or D).
- Your PR description includes a *Risk* section that mentions something the AI did that you weren't sure about.
- Your final code has at least one test that you asked the AI to write *after* reviewing the function.

If you hit all three, you've internalised the loop this notebook teaches.
</details>

### Stretch exercise B — ⭐⭐⭐ Compare two assistants on the same task

Pick one of: VS Code + Copilot, Cursor, Codeium, or your IDE's built-in AI. Then pick a second one. Give them both the *exact same prompt* (something nontrivial — e.g. *"add caching and retry to this function"*). Compare the outputs along five dimensions:

- **Correctness** — does it run? Does it do the right thing?
- **Style fit** — does it match the rest of your code?
- **Verbosity** — minimal change vs sweeping rewrite?
- **Hidden behaviour** — silent error catching, magic defaults?
- **Reviewability** — could a colleague read it in 1 minute?

Write a one-paragraph recommendation: *"For my work I prefer X because Y."*

<details>
<summary>💡 <b>Solution / Answer</b></summary>

There's no objectively right answer — assistants have different strengths and the right choice depends on your work. A *good* comparison has:

- **Concrete observations**, not vague impressions ("Cursor produced 12 lines vs Copilot's 25 for the same prompt; Cursor's was easier to review").
- **At least one negative for the assistant you ultimately recommend** (no tool is uniformly better).
- **A reason that's specific to your context** ("I work on a large monorepo, so multi-file refactoring matters; that pushed me towards Cursor").
</details>

### Stretch exercise C — ⭐⭐⭐ Write a team prompt-style guide

Draft a 1-page *prompt style guide* for a 6-person engineering team. It should cover:

1. **When to use chat vs inline completion.**
2. **Three prompt patterns** the team has agreed to use (e.g. *always paste the existing function before asking for changes*).
3. **Three things the team won't use AI for** (with reasons).
4. **A review checklist** (your version of §4's 60-second checklist).

Style guides are short by design — under one page. The discipline of writing the constraints down is more valuable than the specific contents.

<details>
<summary>💡 <b>Solution / Answer</b></summary>

**Example outline:**

> *"Acme Engineering — Working with AI Assistants"*
>
> **Modes.** Use inline completion for boilerplate (loops, type annotations, repetitive patterns). Use chat for any code that introduces new logic, new dependencies, or touches >1 file.
>
> **Prompt patterns.**
> 1. Always paste the existing function / class before asking for changes (no "update the user-fetch" without showing the user-fetch).
> 2. Ask for one thing per turn.
> 3. Ask for tests in the same turn as the function or the immediately-following turn.
>
> **Won't use AI for.**
> 1. Crypto, auth, or anything in `services/security/`.
> 2. Database migrations (always write by hand, always review with a DBA).
> 3. Code we won't review — if you can't review it, you can't merge it.
>
> **PR review checklist** for AI-generated diffs.
> □ Does it run?  □ Are there tests?  □ Does it match our patterns?  □ No `except Exception: pass`?  □ Scope matches the PR title?

*Real team style guides should be opinionated. A guide that says "use AI sometimes for sometimes-tasks" is useless. Yours should be specific to your context.*
</details>

### Stretch exercise D — ⭐⭐⭐ AI-pair a real Git workflow

Practise the full professional cycle:

1. **Fork** this course's repository on GitHub.
2. Create a feature branch `add-new-stretch-exercise`.
3. With AI assistance, add *one new Stretch exercise* (with skeleton + worked solution + reasoning) to a notebook of your choice. Follow the existing pattern.
4. **Commit** with a meaningful message.
5. **Open a PR** against your own fork's main branch. Write the PR description as in Exercise 3.
6. **Self-review** the PR — comment on lines you'd want to discuss.
7. **Merge** when satisfied.

This is the full motion you'd do at work, just with yourself as the reviewer. Doing it once on a low-stakes project builds the habits that matter when the stakes are higher.


<details>
<summary>💡 <b>Solution / Answer</b></summary>

There's no answer key — the exercise is the doing. If you finished it, you have practised every professional Git operation that matters: branch / commit / push / PR / review / merge, with AI assistance integrated naturally.

**Signs you did it well:**
- Your commit messages read as sentences ("Add Stretch exercise on memoizing decorator to NB 5") rather than fragments ("updates").
- Your PR description explicitly says which Stretch letter was added and why.
- Your self-review comment names at least one thing you'd ask a teammate's opinion on.
- The new exercise follows the existing format (prompt + skeleton + `<details>` solution + Reasoning paragraph) without being told.
</details>

## 🎁 Bonus mini-project — Set up your AI-assisted workflow for *this* course

Spend 30 minutes setting up the workflow you'll use for the rest of this course:

1. **Pick an IDE** (Cursor / VS Code + Copilot / PyCharm + AI). Install it.
2. **Clone** the course repository locally.
3. **Configure the assistant** — pin a prompt or system instruction that says *"This is a teaching repo. Suggest minimal, well-commented Python that matches the existing notebook style. Don't rewrite cells the user didn't ask about."*
4. **Run one of the canonical notebooks** (NB 7 — pandas — is a good one) using the assistant as your study partner. Ask it to explain at least one section in your own words, and to suggest one variation of one exercise.
5. **Note** what worked and what didn't. Adjust your pinned prompt or workflow accordingly.

The pay-off: every subsequent notebook in this course can be *faster* with the assistant — but only if you've set up the workflow up front. Doing it once saves time across the remaining ~20 hours of study.


## 🧠 Key takeaways

- The 2026 baseline IDE is *VS Code + Copilot*, *Cursor*, or *PyCharm + AI*. Pick one, get fluent.
- Nine Git commands cover 90% of professional work. Force-push only your own feature branches.
- Prompt patterns that work: paste context, one ask per turn, be specific about constraints, ask for tests, ask for alternatives.
- Four failure modes of AI code: hallucinated APIs, plausible-but-wrong, architectural drift, silent over-confidence. The 60-second checklist catches most of them.
- Don't use AI for: things you're learning, security-sensitive code, or anything you can't review.

> 🧭 **Back to the mental model.** Every habit in this notebook flows from one picture: the AI is **a fast, confident junior who never says "I don't know."** Briefing patterns (§3) are how you give the junior a good task; the four failure modes and the 60-second checklist (§4) are how you review its work; the *when-not-to* rules (§5) are corollary (1) — *only delegate what you could review.* Hold that one image and the rest is detail.

> 🏢 **Where Meridian stands now.** Meridian's engineer has *built* the 3-tier triage app — briefed the junior function by function, reviewed every diff, caught a silent `except` before it shipped, and merged it through a small reviewed PR. The code runs. But "the code runs" is a POC, not a deployed product: nobody's using it on real tickets, there are no SLOs, no rollback owner, no RACI. Turning *working code* into *an operational reality the organisation can trust* is the final chapter — NB 46.

## ✅ Self-assessment

- I can use my IDE's AI in both completion and chat modes deliberately.
- I can do the nine core Git operations without looking them up.
- I can write a sequence of prompts that produces reviewable code rather than an opaque blob.
- I can name the four failure modes of AI-generated code and detect each one in a diff.
- I can decide when *not* to use the AI assistant.

## 🚀 Next step

→ **NB 46 — BPM integration, governance, and POC → MVP → Production** (`./46_bpm_governance_poc_mvp.ipynb`). The last notebook in Module 14 stitches the methodology together: how AI features fit into existing business processes, how to govern them, and how to take a working prototype all the way to a deployed product. Meridian's triage app becomes Case 1 — followed end-to-end from POC to production.